In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

e:\Anaconda\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [13]:
img_width = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    'dataset/train/train',
    target_size = img_width,
    batch_size = batch_size,
    class_mode = "categorical"
)

val_data = datagen.flow_from_directory(
    'dataset/val',
    target_size = img_width,
    batch_size = batch_size,
    class_mode = "categorical"
)

Found 16854 images belonging to 33 classes.
Found 5824 images belonging to 33 classes.


In [14]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import models, layers
from tensorflow.keras.optimizers import Adam

In [16]:
base_model = MobileNetV2(
    input_shape = (224, 224, 3),
    include_top = False,
    weights = "imagenet"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(train_data.num_classes, activation="softmax")
])

model.compile(
    optimizer = Adam(learning_rate=0.0001),
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

model.fit(train_data, epochs=3, validation_data=val_data)

Epoch 1/3
527/527 ━━━━━━━━━━━━━━━━━━━━ 192s 357ms/step - accuracy: 0.6129 - loss: 1.5897 - val_accuracy: 0.9873 - val_loss: 0.2995
Epoch 2/3
527/527 ━━━━━━━━━━━━━━━━━━━━ 163s 309ms/step - accuracy: 0.9330 - loss: 0.3561 - val_accuracy: 0.9988 - val_loss: 0.0678
Epoch 3/3
527/527 ━━━━━━━━━━━━━━━━━━━━ 164s 311ms/step - accuracy: 0.9738 - loss: 0.1615 - val_accuracy: 0.9993 - val_loss: 0.0274


In [18]:
from tensorflow.keras.preprocessing import image

img = image.load_img("pear.jpg", target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
print(prediction)

pred_class = np.argmax(prediction, axis=1)
print("Predicted class index:", pred_class[0])

class_names = list(train_data.class_indices.keys())
print("Predicted class:", class_names[pred_class[0]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
[[1.11546164e-04 4.27830778e-03 1.76216956e-04 7.21019460e-05
  5.86660719e-03 1.48199048e-04 3.16829653e-03 9.93723734e-05
  1.90445753e-05 8.71211450e-05 6.74992989e-05 5.66250528e-04
  2.09944187e-06 9.68192762e-06 5.09554702e-05 1.65639249e-05
  1.21513003e-04 1.78171399e-06 1.95011773e-04 7.03938349e-05
  1.64386223e-03 2.14394640e-05 9.80517328e-01 9.27305140e-04
  1.61883378e-04 1.14304610e-04 1.50860578e-05 3.30712214e-06
  7.21519638e-04 1.22122037e-05 4.99374873e-04 6.95276176e-05
  1.64266821e-04]]
Predicted class index: 22
Predicted class: Pear


In [7]:
print(train_data.class_indices)

{'Apple Braeburn': 0, 'Apple Granny Smith': 1, 'Apricot': 2, 'Avocado': 3, 'Banana': 4, 'Blueberry': 5, 'Cactus fruit': 6, 'Cantaloupe': 7, 'Cherry': 8, 'Clementine': 9, 'Corn': 10, 'Cucumber Ripe': 11, 'Grape Blue': 12, 'Kiwi': 13, 'Lemon': 14, 'Limes': 15, 'Mango': 16, 'Onion White': 17, 'Orange': 18, 'Papaya': 19, 'Passion Fruit': 20, 'Peach': 21, 'Pear': 22, 'Pepper Green': 23, 'Pepper Red': 24, 'Pineapple': 25, 'Plum': 26, 'Pomegranate': 27, 'Potato Red': 28, 'Raspberry': 29, 'Strawberry': 30, 'Tomato': 31, 'Watermelon': 32}
